# Interactive Dashboard: Exploring Dr. Brooks' Running Performance

This dashboard notebook was extracted from assignment4.ipynb for convenience, to avoid executing the entire original notebook.

I have developed a dashboard for deep exploration.
The dashboard combines four primary filtering:

- a date range
- ground contact time (milliseconds)
- running effectiveness (speed/power)
- running speed
- running cluster (fucused running or other running)

The filtered data is presented in three coordinated scatter plots, colored by running effectiveness.

- Form power vs. ground contact time
- form power vs. speed
- ground contact time vs. speed

These visualizations helps to observe multivariate patterns that a single static analysis could not capture.

In [24]:
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

In [16]:
run_df = pd.read_csv('csv_output/dashboard.csv')

In [17]:
run_df['timestamp'] = pd.to_datetime(run_df['timestamp'])

In [22]:
def add_regression_line(ax, x, y, color="red"):
    mask = (~np.isnan(x)) & (~np.isnan(y))
    x_clean = x[mask]
    y_clean = y[mask]

    slope, intercept = np.polyfit(x_clean, y_clean, 1)

    x_vals = np.linspace(x_clean.min(), x_clean.max(), 100)
    y_vals = slope * x_vals + intercept

    ax.plot(x_vals, y_vals, color=color, linewidth=1, alpha=0.8)

In [18]:
min_date = run_df['timestamp'].min()
max_date = run_df['timestamp'].max()

min_gt = run_df['Ground Time'].min()
max_gt = run_df['Ground Time'].max()

min_re = run_df['Running Effectiveness'].min()
max_re = run_df['Running Effectiveness'].max()

min_speed = run_df['Speed'].min()
max_speed = run_df['Speed'].max()

min_fp = run_df['Form Power'].min()
max_fp = run_df['Form Power'].max()

In [25]:

min_ts = run_df['timestamp'].min()
max_ts = run_df['timestamp'].max()

min_date = min_ts.date()
max_date = max_ts.date()

desc_width = '150px'

days_total = (max_date - min_date).days

date_slider = widgets.IntRangeSlider(
    value=(0, days_total),
    min=0,
    max=days_total,
    step=1,
    description='Date',
    layout={'width': '600px'},
    readout=False,
    style={'description_width': desc_width}
)

start_picker = widgets.DatePicker(
    description='Start',
    layout={'width': '220px'},
    style={'description_width': '80px'}
)
end_picker   = widgets.DatePicker(
    description='End',
    layout={'width': '220px'},
    style={'description_width': '80px'}
)
start_picker.value = min_date
end_picker.value   = max_date

def slider_to_picker(change):
    start_picker.value = int_to_date(date_slider.value[0])
    end_picker.value   = int_to_date(date_slider.value[1])

def picker_to_slider(change):
    if start_picker.value and end_picker.value:
        date_slider.value = (
            (start_picker.value - min_date).days,
            (end_picker.value - min_date).days
        )

def int_to_date(n):
    return min_date + timedelta(days=int(n))

def update_date_readout(change):
    start = int_to_date(date_slider.value[0])
    end   = int_to_date(date_slider.value[1])
    date_slider.readout = f"{start} - {end}"

# update_date_readout(None)
# date_slider.observe(update_date_readout, names='value')
date_slider.observe(slider_to_picker, names='value')
start_picker.observe(picker_to_slider, names='value')
end_picker.observe(picker_to_slider, names='value')

gt_slider = widgets.IntRangeSlider(
    value=(min_gt, max_gt),
    min=min_gt,
    max=max_gt,
    step=0.1,
    description='Ground Time',
    layout={'width': '600px'},
    style={'description_width': desc_width}
)

re_slider = widgets.FloatRangeSlider(
    value=(min_re, max_re),
    min=min_re,
    max=max_re,
    step=0.1,
    description='Running Effectiveness',
    layout={'width': '600px'},
    style={'description_width': desc_width}
)

speed_slider = widgets.FloatRangeSlider(
    value=(min_speed, max_speed),
    min=min_speed,
    max=max_speed,
    step=0.01,
    description='Speed',
    layout={'width': '600px'},
    style={'description_width': desc_width}
)

cluster_label = widgets.HTML(
    "<div style='width:150px; text-align:right;'>Cluster Options</div>"
)
running_cluster = widgets.Checkbox(
    value=True,
    description='Focused Running Cluster',
    style={'description_width': '0px'}
)
other_cluster   = widgets.Checkbox(
    value=True,
    description='Unfocused Running Clusters',
    style={'description_width': desc_width}
)
cluster_ui = widgets.VBox([
    widgets.HBox([cluster_label, running_cluster]),
    widgets.HBox([widgets.Label(""), other_cluster])
])

def show_dashboard(start_date, end_date,
                   gt_range, re_range, speed_range,
                   use_cluster2, use_other):

    # start_date = int_to_date(date_range[0])
    # end_date   = int_to_date(date_range[1])
    
    filtered_df = run_df.copy()

    filtered_df = filtered_df[
        (filtered_df['timestamp'].dt.date >= start_date) &
        (filtered_df['timestamp'].dt.date <= end_date)
    ]

    filtered_df = filtered_df[
        (filtered_df['Ground Time'] >= gt_range[0]) &
        (filtered_df['Ground Time'] <= gt_range[1])
    ]

    filtered_df = filtered_df[
        (filtered_df['Running Effectiveness'] >= re_range[0]) &
        (filtered_df['Running Effectiveness'] <= re_range[1])
    ]

    filtered_df = filtered_df[
        (filtered_df['Speed'] >= speed_range[0]) &
        (filtered_df['Speed'] <= speed_range[1])
    ]

    # cluster filter
    if use_cluster2 and not use_other:
        filtered_df = filtered_df[filtered_df['cluster'] == 2]
    elif use_other and not use_cluster2:
        filtered_df = filtered_df[filtered_df['cluster'] != 2]
    # if both True or both False -> no filtering

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Scatter Plot Grid (color varies by metric)", fontsize=16)

    alpha = 0.5

    ffp = filtered_df['Form Power']
    fgt = filtered_df["Ground Time"]
    fre = filtered_df["Running Effectiveness"]
    fes = filtered_df['Speed']
    fcc = filtered_df['Cadence']

    # 1. FP vs GT
    ax = axs[0]
    sc = ax.scatter(ffp, fgt, c=fre, s=2, alpha=alpha, cmap='Blues', vmin=min_re, vmax=max_re)
    add_regression_line(ax, ffp, fgt)
    ax.set_title("Form Power vs Ground Time\n(color: RE)")
    ax.set_xlabel("Form Power (W)")
    ax.set_xlim(min_fp - (max_fp - min_fp) * 0.05, max_fp + (max_fp - min_fp) * 0.05)
    ax.set_ylabel("Ground Time (ms)")
    ax.set_ylim(min_gt - (max_gt - min_gt) * 0.05, max_gt + (max_gt - min_gt) * 0.05)
    fig.colorbar(sc, ax=ax)

    # 2. FP vs Speed
    ax = axs[1]
    sc = ax.scatter(ffp, fes, c=fre, s=2, alpha=alpha, cmap='Blues', vmin=min_re, vmax=max_re)
    add_regression_line(ax, ffp, fes)
    ax.set_title("Form Power vs Speed\n(color: RE)")
    ax.set_xlabel("Form Power (W)")
    ax.set_xlim(min_fp - (max_fp - min_fp) * 0.05, max_fp + (max_fp - min_fp) * 0.05)
    ax.set_ylabel("Speed (m/s)")
    ax.set_ylim(min_speed - (max_speed - min_speed) * 0.05, max_speed + (max_speed - min_speed) * 0.05)
    fig.colorbar(sc, ax=ax)

    # 3. GT vs Speed
    ax = axs[2]
    sc = ax.scatter(fgt, fes, c=fre, s=2, alpha=alpha, cmap='Blues', vmin=min_re, vmax=max_re)
    add_regression_line(ax, fgt, fes)
    ax.set_title("Ground Time vs Speed\n(color: RE)")
    ax.set_xlabel("Ground Time (ms)")
    ax.set_xlim(min_gt - (max_gt - min_gt) * 0.05, max_gt + (max_gt - min_gt) * 0.05)
    ax.set_ylabel("Speed (m/s)")
    ax.set_ylim(min_speed - (max_speed - min_speed) * 0.05, max_speed + (max_speed - min_speed) * 0.05)
    fig.colorbar(sc, ax=ax)

    plt.tight_layout()
    plt.show()


ui = widgets.VBox([
    date_slider,
    widgets.HBox([
        widgets.HTML("<div style='width:150px; text-align:right;'></div>"),
        start_picker,
        end_picker
    ]),
    gt_slider,
    re_slider,
    speed_slider,
    cluster_ui
])

def get_date_range():
    return (start_picker.value, end_picker.value)

out = widgets.interactive_output(
    show_dashboard,
    {
        'start_date': start_picker,
        'end_date': end_picker,
        'gt_range': gt_slider,
        're_range': re_slider,
        'speed_range': speed_slider,
        'use_cluster2': running_cluster,
        'use_other': other_cluster
    }
)

display(ui, out)

Output()